# Example: Rebalancing Engine and Scorecard

In this example, we turn the Cobb-Douglas allocator from the previous notebook into a production-style engine and measure what it buys us. Three deliverables:

* __Wire the allocator into a rebalancing engine:__ Drop the **Cobb-Douglas utility allocator** into a daily rebalancing loop with trigger rules (drawdown limit, turnover cap, reallocation schedule) and run it on a single synthetic forward path.
* __Produce a four-row strategy scorecard:__ Compare the engine against the Session 1 min-var portfolio, an equal-weight buy-and-hold, and a risk-free baseline along terminal wealth, Sharpe, volatility, and maximum drawdown.
* __Sweep the constant elasticity of substitution (CES) elasticity parameter $\sigma$:__ Rerun the engine with four CES elasticity values plus an adaptive-elasticity variant to see how *conviction in the top-scoring asset* tunes long-run performance on this path.

The engine, the scorecard, and the elasticity sweep together frame the question the next notebook asks on 5,000 paths.

> __Learning Objectives:__
>
> * __Run the Cobb-Douglas rebalancing engine:__ Execute the rebalancing engine with production-style trigger rules including drawdown limits and turnover caps. Observe how tighter or looser drawdown thresholds affect capital protection and recovery.
> * __Produce a strategy scorecard:__ Build a four-row scorecard comparing the AI engine to the Session 1 min-var portfolio, an equal-weight buy-and-hold, and a risk-free benchmark. Evaluate return, volatility, Sharpe ratio, and maximum drawdown across strategies.
> * __Sweep CES elasticity inside the engine:__ Run the rebalancing engine with four CES elasticity values plus an adaptive elasticity variant, and compare wealth trajectories and scorecard metrics. Identify which end of the elasticity axis rewards conviction versus diversification on this path.

Let's dive in!

___

## Setup, Data and Prerequisites

In [1]:
include("Include.jl");

### Constants


In [2]:
# Scorecard configuration
ENGINE_RUN_DATA_PATH = joinpath(_PATH_TO_DATA, "engine-run-data.jld2")  # path to Example 1 hand-off file
Δt = 1.0 / 252.0              # trading-day step (years)
L_short = 21                  # short EMA window (days), must match Example 1
L_long = 63                   # long EMA window (days), must match Example 1
offset = L_short + L_long     # warmup offset before trading begins (days)
n_trading_days = 252          # trading horizon after warmup (days)

252

Load the data produced by the `BuildCobbDouglasAllocator` notebook (Example 1 in this session). In the code block below, we read `engine-run-data.jld2` and return the following bindings: `my_tickers::Vector{String}`, `price_matrix::Matrix{Float64}`, `market_prices::Vector{Float64}`, `lambda_series::Vector{Float64}`, `gm_ema::Vector{Float64}`, `sim_params::Dict{String,Tuple{Float64,Float64,Float64}}`, `context::MyRebalancingContextModel`, the three baseline wealth series `minvar_wealth::Vector{Float64}`, `equalweight_wealth::Vector{Float64}`, `riskfree_wealth::Vector{Float64}`, and the derived scalars `g_f::Float64`, `B₀::Float64`, `N::Int`.

In [3]:
my_tickers, price_matrix, lambda_series, market_prices, gm_ema, sim_params, context, minvar_wealth, equalweight_wealth, riskfree_wealth, g_f, B₀, N = let
    # --- Step 1: Load saved data from Example 1 ---
    data = load_results(ENGINE_RUN_DATA_PATH);

    # --- Step 2: Extract market and engine data into top-level bindings ---
    my_tickers         = data["my_tickers"]::Vector{String};
    price_matrix       = data["price_matrix"]::Matrix{Float64};
    lambda_series      = data["lambda_series"]::Vector{Float64};
    market_prices      = data["market_prices"]::Vector{Float64};
    gm_ema             = data["gm_ema"]::Vector{Float64};
    sim_params         = data["sim_params"]::Dict{String,Tuple{Float64,Float64,Float64}};
    context            = data["context"]::MyRebalancingContextModel;
    minvar_wealth      = Float64.(data["minvar_wealth"]);
    equalweight_wealth = Float64.(data["equalweight_wealth"]);
    riskfree_wealth    = Float64.(data["riskfree_wealth"]);
    g_f                = Float64(data["g_f"]);

    # --- Step 3: Derived dimensions ---
    B₀             = context.B;            # initial budget from the context
    N              = length(my_tickers);   # number of assets in the portfolio

    println("Loaded engine data: $(N) tickers, $(n_trading_days) trading days after $(offset) warmup")
    println("  Tickers: $(my_tickers)")
    println("  g_f (continuous, %/yr): $(round(g_f*100, digits=2))")
    my_tickers, price_matrix, lambda_series, market_prices, gm_ema, sim_params, context, minvar_wealth, equalweight_wealth, riskfree_wealth, g_f, B₀, N
end


Loaded engine data: 20 tickers, 252 trading days after 84 warmup


  Tickers: ["VZ", "T", "MCD", "PG", "KO", "PEP", "WMT", "XOM", "CVX", "JPM", "BRK.B", "JNJ", "MRK", "HON", "UPS", "AAPL", "MSFT", "APD", "AMT", "NEE"]
  g_f (continuous, %/yr): 4.5


(["VZ", "T", "MCD", "PG", "KO", "PEP", "WMT", "XOM", "CVX", "JPM", "BRK.B", "JNJ", "MRK", "HON", "UPS", "AAPL", "MSFT", "APD", "AMT", "NEE"], [1.0 40.73 … 175.57 80.28; 2.0 40.91225279856964 … 175.24957010095764 80.01054497614588; … ; 335.0 35.483936120769435 … 160.22403328185604 79.86155644336469; 336.0 35.393240790856424 … 159.50891302473136 78.9286634674355], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.14004160179568292, 0.14891543640320415, 0.14507441903268914, 0.13620626965207605, 0.13770701432429533, 0.11712203196430293, 0.1220393754348914, 0.11743922562327591, 0.11085443482436341, 0.10896863119704392], [100.0, 99.69778731908029, 99.41484728083054, 98.7569844749604, 99.01570993384115, 99.79427860430765, 99.69984480419191, 99.72958712904887, 98.87257428992179, 99.07615629317863  …  87.5952954549071, 87.02072317631172, 88.70846473765612, 89.46943760548004, 88.01556721669371, 91.24198317852776, 87.7349808578427, 89.02327846413655, 89.35310431468216, 88.71930500626051], [

### Implementations
The `scorecard_metrics` function computes performance metrics from a wealth time series. It takes a `wealth::Vector{Float64}` vector of daily wealth values and a `label::String` strategy name, and returns a tuple of `(label, total_return, volatility, sharpe_ratio, max_drawdown)` with return, volatility, and drawdown expressed as percentages. Volatility and Sharpe are computed from **daily simple returns** (not continuously-compounded growth rates) because that is the convention the Sharpe ratio is defined in.

In [4]:
"""
    scorecard_metrics(wealth::Array{Float64,1}, label::String) -> Tuple

Compute performance metrics from a daily wealth time series.

### Arguments
- `wealth::Array{Float64,1}`: daily wealth values (e.g., portfolio value over time).
- `label::String`: strategy name used to tag the output row.

### Returns
A tuple `(label, total_return, volatility, sharpe_ratio, max_drawdown)` where return, volatility,
and max drawdown are expressed as percentages.
"""
function scorecard_metrics(wealth::Array{Float64,1}, label::String)

    # --- Step 1: Compute daily simple returns (Sharpe/vol convention) ---
    daily_returns = diff(wealth) ./ wealth[1:end-1];

    # --- Step 2: Compute cumulative return (%) ---
    total_return = (wealth[end] / wealth[1] - 1.0) * 100;

    # --- Step 3: Compute annualized volatility (%) ---
    vol = std(daily_returns) * sqrt(252) * 100;

    # --- Step 4: Compute Sharpe ratio (return / vol) ---
    # Guard against deterministic series (e.g. continuously-compounded risk-free).
    # std of a constant stream is numerically ~1e-15 rather than exactly zero, so
    # `vol > 0` is not strict enough — we need a tolerance well above FP noise but
    # well below any real asset's annualized vol.
    sharpe = vol > 1e-6 ? total_return / vol : 0.0;

    # --- Step 5: Compute maximum drawdown (%) ---
    peak = accumulate(max, wealth);
    dd = maximum((peak .- wealth) ./ peak) * 100;

    # --- Step 6: Return rounded metrics ---
    return (label, round(total_return, digits=2), round(vol, digits=2),
        round(sharpe, digits=2), round(dd, digits=2))
end;

___
## Task 1: Cobb-Douglas Rebalancing Engine with Trigger Rules
In this task, we run the Cobb-Douglas utility allocator inside the full rebalancing engine with realistic trigger rules: a 15% drawdown limit (circuit breaker) and a 50% turnover cap (trading cost control). We compare three drawdown thresholds to see how the safety net affects performance.

> __What should we see?__
>
> Tighter drawdown limits (10%) cause the engine to de-risk to cash earlier and more often, protecting capital but potentially missing recoveries. Looser limits (25%) allow more volatility. The engine uses Cobb-Douglas utility to decide _how_ to allocate; the trigger rules decide _whether_ to allocate.

The code below sweeps three drawdown limits using [the `run_rebalancing_engine(...)` function](../../code/docs/build/session2.html), and the results are stored in the `dd_wealth_curves::Dict{Float64, Array{Float64,1}}` variable.

In [5]:
dd_wealth_curves = let
    # --- Step 1: Define drawdown thresholds and plot styling ---
    drawdown_limits = [0.10, 0.15, 0.25];                  # three drawdown trigger levels to compare
    colors          = [:red :orange :green];                # color per threshold
    labels          = ["DD ≤ 10%", "DD ≤ 15%", "DD ≤ 25%"]; # legend labels

    # Storage: maps each drawdown limit to its wealth curve
    dd_wealth_curves = Dict{Float64, Array{Float64,1}}();

    # --- Step 2: Initialize the plot ---
    p = plot(size=(800, 450), title="Cobb-Douglas Engine: Drawdown-Triggered De-Risking",
        xlabel="Trading Day (after warmup)", ylabel="Wealth (\$)", legend=:topleft)

    # --- Step 3: Sweep drawdown limits ---
    for (j, dd_limit) ∈ enumerate(drawdown_limits)

        # Build trigger rules: drawdown limit varies, turnover cap fixed at 50%
        rules = build(MyTriggerRules, (
            max_drawdown = dd_limit,
            max_turnover = 0.50,
            rebalance_schedule = ones(Int, n_trading_days)  # rebalance every day
        ));

        # Run the rebalancing engine with Cobb-Douglas allocator
        results = run_rebalancing_engine(context, rules, lambda_series;
            offset=offset, allocator=:cobb_douglas);

        # Convert allocation results into a wealth time series
        wealth = compute_wealth_series(results, price_matrix, my_tickers; offset=offset);
        dd_wealth_curves[dd_limit] = wealth;  # store for later use

        # Plot this drawdown scenario
        plot!(p, 1:length(wealth), wealth, label=labels[j], linewidth=2, color=colors[j])
    end

    # --- Step 4: Overlay the Session 1 min-var and equal-weight baselines ---
    plot!(p, 1:length(minvar_wealth), minvar_wealth,
        label="S1 Min-Var (buy-and-hold)", linewidth=1.5, color=:gray20, linestyle=:dash)
    plot!(p, 1:length(equalweight_wealth), equalweight_wealth,
        label="Equal-Weight (buy-and-hold)", linewidth=1.5, color=:gray60, linestyle=:dashdot)
    p
    dd_wealth_curves
end


Dict{Float64, Vector{Float64}} with 3 entries:
  0.15 => [10000.0, 9987.73, 9985.73, 9986.39, 9987.48, 9988.97, 9990.72, 9992.…
  0.1  => [10000.0, 9987.73, 9985.73, 9986.39, 9987.48, 9988.97, 9990.72, 9992.…
  0.25 => [10000.0, 9987.73, 9985.73, 9986.39, 9987.48, 9988.97, 9990.72, 9992.…

___
## Task 2: Scorecard, Engine vs. Baselines
In this task, we produce a scorecard comparing three strategies: the AI Cobb-Douglas engine (DD $\leq$ 15%, $\tau \leq$ 50%), equal-weight buy-and-hold, and risk-free. The scorecard tracks return, volatility, Sharpe ratio, and maximum drawdown.

> __What should we see?__
>
> The engine should show better risk-adjusted performance (lower drawdown, potentially better Sharpe) than static allocations, at the cost of higher trading activity. The scorecard quantifies exactly how much adaptability costs and what it buys.

In the code block below, we run the engine via [the `run_rebalancing_engine(...)` function](../../code/docs/build/session2.html), compute per-strategy metrics, print the scorecard, and return `engine_wealth::Vector{Float64}` (the AI engine's daily wealth trajectory) for use in downstream plots.

In [6]:
engine_wealth = let
    # --- Step 1: Run the moderate engine configuration (DD ≤ 15%, τ ≤ 50%) ---
    rules = build(MyTriggerRules, (
        max_drawdown = 0.15, max_turnover = 0.50,
        rebalance_schedule = ones(Int, n_trading_days)
    ));
    results = run_rebalancing_engine(context, rules, lambda_series;
        offset=offset, allocator=:cobb_douglas);
    engine_wealth = compute_wealth_series(results, price_matrix, my_tickers; offset=offset);

    # --- Step 2: Compute metrics for each strategy ---
    rf_label = "Risk-Free ($(round(g_f*100, digits=2))%)";
    eng = scorecard_metrics(engine_wealth,      "Cobb-Douglas Engine (DD≤15%, τ≤50%)");
    mv  = scorecard_metrics(minvar_wealth,      "S1 Min-Var (buy-and-hold)");
    eqw = scorecard_metrics(equalweight_wealth, "Equal-Weight (buy-and-hold)");
    rf  = scorecard_metrics(riskfree_wealth,    rf_label);

    # --- Step 3: Build and display the scorecard table ---
    scorecard = DataFrame(
        "Strategy"         => [eng[1], mv[1], eqw[1], rf[1]],
        "Return (%)"       => [eng[2], mv[2], eqw[2], rf[2]],
        "Volatility (%)"   => [eng[3], mv[3], eqw[3], rf[3]],
        "Sharpe Ratio"     => [eng[4], mv[4], eqw[4], rf[4]],
        "Max Drawdown (%)" => [eng[5], mv[5], eqw[5], rf[5]]
    );

    println("═"^70)
    println("  SESSION 2 SCORECARD: Cobb-Douglas Engine vs. S1 + Baselines")
    println("═"^70)
    pretty_table(scorecard; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))

    # --- Step 4: Save results for downstream sessions ---
    save_results(joinpath(_PATH_TO_DATA, "session2-scorecard.jld2"), Dict(
        "scorecard"          => scorecard,
        "engine_wealth"      => engine_wealth,
        "minvar_wealth"      => minvar_wealth,
        "equalweight_wealth" => equalweight_wealth,
        "riskfree_wealth"    => riskfree_wealth,
    ));
    engine_wealth
end


══════════════════════════════════════════════════════════════════════


  SESSION 2 SCORECARD: Cobb-Douglas Engine vs. S1 + Baselines
══════════════════════════════════════════════════════════════════════
 ------------------------------------- ------------ ---------------- -------------- ------------------
                             Strategy   Return (%)   Volatility (%)   Sharpe Ratio   Max Drawdown (%) 
                               String      Float64          Float64        Float64            Float64 
 ------------------------------------- ------------ ---------------- -------------- ------------------
  Cobb-Douglas Engine (DD≤15%, τ≤50%)        28.11            10.68           2.63               3.62
            S1 Min-Var (buy-and-hold)         4.89            16.84           0.29              19.81
          Equal-Weight (buy-and-hold)        -6.28            16.24          -0.39              20.79
                     Risk-Free (4.5%)         4.58              0.0            0.0                0.0
 ------------------------------------- --------

253-element Vector{Float64}:
 10000.0
  9987.731215713717
  9985.727676670544
  9986.38729162244
  9987.477090918732
  9988.969590594384
  9990.715292438359
  9992.689130669381
  9993.00761432021
 10054.3307152072
 10078.976132014719
 10115.732193688498
 10153.262524631673
     ⋮
 12217.083749683256
 12216.079820889034
 12212.530572146787
 12210.68673046697
 12404.1167384966
 12492.915702760754
 12471.143875748168
 12875.452449433113
 12683.60758194113
 12807.325997773858
 12862.957056754401
 12810.75486522993

The code below plots the wealth curves for all four strategies (engine, S1 min-var, equal-weight, risk-free) on a single axis.

In [7]:
let
    # --- Step 1: Define the x-axis (trading days after warmup) ---
    days = 1:length(engine_wealth);

    # --- Step 2: Plot each strategy ---
    plot(days, engine_wealth,       label="Cobb-Douglas Engine", linewidth=2.5, color=:steelblue)
    plot!(days, minvar_wealth,      label="S1 Min-Var",           linewidth=2,   color=:coral,    linestyle=:dash)
    plot!(days, equalweight_wealth, label="Equal-Weight",         linewidth=2,   color=:green,    linestyle=:dashdot)
    plot!(days, riskfree_wealth,    label="Risk-Free",            linewidth=1.5, color=:gray50,   linestyle=:dot)

    # --- Step 3: Add axis labels and formatting ---
    xlabel!("Trading Day (after warmup)")
    ylabel!("Wealth (\$)")
    title!("Session 2: Cobb-Douglas Rebalancing Engine vs. Baselines")
    plot!(size=(800, 450), legend=:topleft)
end

print device already activated
print device already activated


___
## Task 3: CES Elasticity Sweep — How Concentration Tunes Engine Behavior
In this task, we swap the allocator inside the rebalancing engine from Cobb-Douglas to CES utility and run the engine at four elasticity values: $\sigma \in \{0.5, 1.0, 2.0, 3.0\}$. CES with $\sigma = 1$ is the Cobb-Douglas limit (same allocation, same wealth path); $\sigma > 1$ concentrates more budget in the single best-scoring asset, $\sigma < 1$ diversifies more aggressively. By holding trigger rules and sentiment fixed, the σ sweep isolates the effect of *how much* the allocator reacts to its own preference weights.

> __What should we see?__
>
> The $\sigma = 1$ CES run should overlay the Cobb-Douglas engine almost exactly (they're the same allocation — the matching is a sanity check on the library's CES path). As $\sigma$ grows, wealth trajectories diverge from Cobb-Douglas: high-$\sigma$ runs place larger bets on the top-γ asset each day, which amplifies upside in trending regimes and downside in choppy ones. Low-$\sigma$ runs look smoother but typically underperform because they spread budget even across weakly-preferred assets.

In the code block below, we run the rebalancing engine four times (once per σ value), each with the same trigger rules (DD ≤ 15%, τ ≤ 50%) and the same $\lambda$ series from Task 1. The per-σ wealth series are stored in the `ces_wealth_curves::Dict{Float64,Vector{Float64}}` dictionary.

In [8]:
ces_wealth_curves = let
    # --- Step 1: Define the σ grid ---
    # σ = 1 is the Cobb-Douglas limit (sanity check row).
    σ_values = [0.5, 1.0, 2.0, 3.0];

    # --- Step 2: Trigger rules and sentiment shared across all runs ---
    rules = build(MyTriggerRules, (
        max_drawdown = 0.15, max_turnover = 0.50,
        rebalance_schedule = ones(Int, n_trading_days)
    ));

    # --- Step 3: Run the engine once per σ and store the wealth series ---
    ces_wealth_curves = Dict{Float64, Vector{Float64}}();
    for σ ∈ σ_values
        results_σ = run_rebalancing_engine(context, rules, lambda_series;
            offset = offset, allocator = :ces, sigma = σ);
        wealth_σ = compute_wealth_series(results_σ, price_matrix, my_tickers; offset = offset);
        ces_wealth_curves[σ] = wealth_σ;
    end

    println("CES engine runs complete for σ ∈ $(σ_values)")
    for σ ∈ σ_values
        w = ces_wealth_curves[σ];
        println("  σ=$(σ):  W/W₀ final = $(round(w[end]/B₀, digits=3)),  max = $(round(maximum(w)/B₀, digits=3))")
    end
    ces_wealth_curves
end


CES engine runs complete for σ ∈ [0.5, 1.0, 2.0, 3.0]
  σ=0.5:  W/W₀ final = 1.276,  max = 1.283
  σ=1.0:  W/W₀ final = 1.281,  max = 1.288
  σ=2.0:  W/W₀ final = 1.292,  max = 1.297
  σ=3.0:  W/W₀ final = 1.303,  max = 1.309


Dict{Float64, Vector{Float64}} with 4 entries:
  2.0 => [10000.0, 9987.73, 9985.73, 9986.39, 9987.48, 9988.97, 9990.72, 9992.6…
  0.5 => [10000.0, 9987.73, 9985.73, 9986.39, 9987.48, 9988.97, 9990.72, 9992.6…
  3.0 => [10000.0, 9987.73, 9985.73, 9986.39, 9987.48, 9988.97, 9990.72, 9992.6…
  1.0 => [10000.0, 9987.73, 9985.73, 9986.39, 9987.48, 9988.97, 9990.72, 9992.6…

The code below plots the four CES wealth trajectories on a single axis, with the Cobb-Douglas engine and the S1 min-var baseline as reference lines.

In [9]:
let
    days = 1:length(engine_wealth);
    σ_values = sort(collect(keys(ces_wealth_curves)));
    palette = [:steelblue, :coral, :goldenrod, :firebrick];

    # --- Step 1: Plot the four CES runs ---
    p = plot(size=(800, 450), legend=:topleft,
        title="CES σ Sweep: Engine Wealth vs. Elasticity of Substitution",
        xlabel="Trading Day (after warmup)", ylabel="Wealth (\$)")
    for (k, σ) ∈ enumerate(σ_values)
        plot!(p, days, ces_wealth_curves[σ],
            label="CES σ=$(σ)", linewidth=2, color=palette[k])
    end

    # --- Step 2: Overlay the Cobb-Douglas engine and S1 min-var for reference ---
    plot!(p, days, engine_wealth,
        label="Cobb-Douglas Engine", linewidth=2.5, color=:black, linestyle=:dash)
    plot!(p, days, minvar_wealth,
        label="S1 Min-Var (buy-and-hold)", linewidth=1.5, color=:gray50, linestyle=:dot)
    p
end

print device already activated
print device already activated
print device already activated


The code below extends the Task 2 scorecard with one row per CES run, so return, volatility, Sharpe, and drawdown are directly comparable across σ values and against the Cobb-Douglas engine and baselines.

In [10]:
let
    # --- Step 1: Metrics for each CES run ---
    σ_values = sort(collect(keys(ces_wealth_curves)));
    ces_rows = [scorecard_metrics(ces_wealth_curves[σ], "CES Engine (σ=$(σ))") for σ ∈ σ_values];

    # --- Step 2: Append to the existing four-row baseline scorecard ---
    rf_label = "Risk-Free ($(round(g_f*100, digits=2))%)";
    eng = scorecard_metrics(engine_wealth,      "Cobb-Douglas Engine (DD≤15%, τ≤50%)");
    mv  = scorecard_metrics(minvar_wealth,      "S1 Min-Var (buy-and-hold)");
    eqw = scorecard_metrics(equalweight_wealth, "Equal-Weight (buy-and-hold)");
    rf  = scorecard_metrics(riskfree_wealth,    rf_label);

    all_rows = vcat([eng], ces_rows, [mv, eqw, rf]);

    # --- Step 3: Build the combined DataFrame ---
    df = DataFrame(
        "Strategy"         => [r[1] for r ∈ all_rows],
        "Return (%)"       => [r[2] for r ∈ all_rows],
        "Volatility (%)"   => [r[3] for r ∈ all_rows],
        "Sharpe Ratio"     => [r[4] for r ∈ all_rows],
        "Max Drawdown (%)" => [r[5] for r ∈ all_rows]
    );

    println("═"^80)
    println("  SESSION 2 EXTENDED SCORECARD: CES Elasticity Sweep")
    println("═"^80)
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end

════════════════════════════════════════════════════════════════════════════════
  SESSION 2 EXTENDED SCORECARD: CES Elasticity Sweep
════════════════════════════════════════════════════════════════════════════════
 ------------------------------------- ------------ ---------------- -------------- ------------------
                             Strategy   Return (%)   Volatility (%)   Sharpe Ratio   Max Drawdown (%) 
                               String      Float64          Float64        Float64            Float64 
 ------------------------------------- ------------ ---------------- -------------- ------------------
  Cobb-Douglas Engine (DD≤15%, τ≤50%)        28.11            10.68           2.63               3.62
                   CES Engine (σ=0.5)        27.62            10.65           2.59               3.44
                   CES Engine (σ=1.0)        28.11            10.68           2.63               3.62
                   CES Engine (σ=2.0)        29.16             10.6

The code below identifies the best and worst σ value in the sweep by final `W/W₀`, highlighting the point on the elasticity axis where the engine wins (or loses) the most.

In [11]:
let
    # --- Step 1: Collect terminal W/W₀ by σ ---
    σ_values = sort(collect(keys(ces_wealth_curves)));
    terminals = [(σ, ces_wealth_curves[σ][end] / B₀) for σ ∈ σ_values];

    # --- Step 2: Sort by terminal wealth ---
    sort!(terminals, by = x -> x[2], rev = true);

    # --- Step 3: Report ---
    (best_σ, best_w)   = terminals[1];
    (worst_σ, worst_w) = terminals[end];
    println("Best  σ: $(best_σ)   → W/W₀ = $(round(best_w, digits=3))")
    println("Worst σ: $(worst_σ)  → W/W₀ = $(round(worst_w, digits=3))")
    println()
    println("Full ranking (σ → W/W₀):")
    for (σ, w) ∈ terminals
        println("  σ=$(σ):  $(round(w, digits=3))")
    end
end

Best  σ: 3.0   → W/W₀ = 1.303


Worst σ: 0.5  → W/W₀ = 1.276

Full ranking (σ → W/W₀):
  σ=3.0:  1.303
  σ=2.0:  1.292
  σ=1.0:  1.281
  σ=0.5:  1.276


### Adaptive σ: Letting the Engine Choose Its Own Elasticity

The static sweep above holds σ constant across the entire trading horizon. But the [adaptive elasticity model](eCornell-AI-Finance-S2-Lecture-AI-RebalancingEngine-May-2026.ipynb) links σ to the sentiment signal λ: concentrate when neutral, diversify when sentiment is extreme. The engine computes $\sigma(\lambda_t) = \sigma_{\min} + (\sigma_{\max} - \sigma_{\min})/(1 + |\lambda_t|)$ at each rebalance day using [the `compute_adaptive_sigma(...)` function](../../code/docs/build/session2.html).

In the code block below, we run the engine with `adaptive_sigma=true` and return `adaptive_wealth::Vector{Float64}` (the wealth trajectory under adaptive σ) and `sigma_series::Vector{Float64}` (the per-day σ values). A two-panel figure overlays the adaptive run on the static CES sweep (top) and shows the σ trajectory (bottom).

In [12]:
adaptive_wealth, sigma_series = let
    # --- Step 1: Run the CES engine with adaptive σ(λ) ---
    rules = build(MyTriggerRules, (
        max_drawdown = 0.15, max_turnover = 0.50,
        rebalance_schedule = ones(Int, n_trading_days)
    ));
    results_adaptive = run_rebalancing_engine(context, rules, lambda_series;
        offset = offset, allocator = :ces, adaptive_sigma = true, sigma_bounds = (0.5, 5.0));
    adaptive_wealth = compute_wealth_series(results_adaptive, price_matrix, my_tickers; offset = offset);

    # --- Step 2: Extract the per-day σ series for plotting ---
    trading_days = sort(collect(keys(results_adaptive)));
    sigma_series = [results_adaptive[d].sigma for d in trading_days];

    # --- Step 3: Two-panel plot: wealth comparison (top) + σ(t) trajectory (bottom) ---
    days = 1:length(adaptive_wealth);
    σ_values = sort(collect(keys(ces_wealth_curves)));
    palette = [:steelblue, :coral, :goldenrod, :firebrick];

    p1 = plot(size = (800, 300), legend = :topleft,
        title = "Adaptive σ(λ) vs. Static CES Sweep",
        ylabel = "W/W₀")
    for (k, σ) ∈ enumerate(σ_values)
        plot!(p1, days, ces_wealth_curves[σ] ./ B₀,
            label = "Static σ=$(σ)", linewidth = 1.5, color = palette[k], alpha = 0.5)
    end
    plot!(p1, days, adaptive_wealth ./ B₀,
        label = "Adaptive σ(λ)", linewidth = 2.5, color = :black)

    p2 = plot(0:length(sigma_series)-1, sigma_series,
        label = "", linewidth = 1.5, color = :black,
        xlabel = "Trading Day (after warmup)", ylabel = "σ(λₜ)",
        title = "Elasticity Trajectory", size = (800, 200))
    hline!(p2, [0.5, 5.0], linestyle = :dot, color = :gray50, label = "")

    plot(p1, p2, layout = grid(2, 1, heights = [0.65, 0.35]), size = (800, 500))
    adaptive_wealth, sigma_series
end


([10000.0, 9987.731215713717, 9985.727676670544, 9986.38729162244, 9987.477090918732, 9988.969590594384, 9990.715292438359, 9992.689130669381, 9993.00761432021, 10016.729110083557  …  12806.443428876739, 12804.59958719692, 12938.14935057904, 13016.928147358843, 13019.366714719854, 13128.44676989058, 13093.2817826999, 13179.136453164407, 13228.458700868487, 13159.598894526569], [5.0, 4.910895834671338, 4.790049232927193, 4.702751135545949, 4.637952598894759, 4.575975864558536, 4.529532753207051, 4.48908294095547, 4.454255017119998, 4.453422070105599  …  4.44722437577018, 4.416737348475102, 4.42987558293496, 4.460548467469705, 4.455324124175002, 4.52820808402407, 4.5105544408865725, 4.527064646392763, 4.5509358012433845, 4.557824426595913])

___
## Summary
This example ran the Cobb-Douglas rebalancing engine with production-style trigger rules on the single hybrid Single Index Model (SIM) path generated in Example 1, compared it against the Session 1 min-var portfolio plus equal-weight and risk-free baselines via a four-row scorecard, swapped the allocator to CES utility with four static elasticity values, and then ran the engine with adaptive elasticity to let the substitution parameter respond to market sentiment.

> __Key Takeaways:__
>
> * __Drawdown limits act as circuit breakers:__ Tighter drawdown thresholds cause the engine to de-risk to cash earlier, protecting capital at the cost of missing recoveries. Looser thresholds allow more volatility but capture more upside.
> * __CES elasticity tunes conviction:__ The same preference weights drive different wealth paths depending on the elasticity value. Higher elasticity concentrates budget in the top-ranked asset and amplifies both gains and losses.
> * __Adaptive elasticity adds a second adaptive channel:__ Linking the elasticity to the sentiment signal lets the engine concentrate when sentiment is neutral and diversify when sentiment is extreme. Session 3 revisits this choice using multi-armed bandits that learn the best elasticity per regime from data.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The rebalancing engine described here is a pedagogical tool using synthetic data and simplified models. Real-world algorithmic trading involves regulatory requirements, execution risk, and operational considerations beyond the scope of this session.

___